Ссылка на урок: [Спринт 2/5: 2 RAG: от сырых данных до точных ответов → Тема 2/4: Векторные базы и индексы → Урок 4/5: Оптимизация эмбеддингов: размерность, PQ и scalar quantization]( https://practicum.yandex.ru/learn/ai-engineer/courses/99ad0dee-d9e6-4da1-b921-64e7e9653a4e/sprints/1104074/topics/7c1be2cd-4a52-4f8a-8dce-db7d2e89a79d/lessons/3c407459-5eec-4223-86d8-2de32f7c5336/#ed84840e-0134-40b4-9d29-0dca886301cd )
Часть 2

In [2]:
import os                                                 # Размер файла индекса на диске
import time                                               # Замер времени поиска
import numpy as np                                        # Матрицы эмбеддингов
import faiss                                              # Индексы и поиск по векторам
from datasets import load_dataset                         # Скачивание корпуса MIRACL
from sentence_transformers import SentenceTransformer     # Эмбеддер на GPU

# --- Корпус: стримим русский MIRACL, берём пассажи разумной длины ---
# streaming=True не качает все 9.5 млн пассажей, а отдаёт их по одному
# Грузим json-шарды корпуса напрямую: datasets 4.x больше не поддерживает
# loading-скрипты, поэтому короткий путь load_dataset("miracl/miracl-corpus", "ru")
# не работает — указываем файлы данных явно через data_files.
stream = load_dataset(
    "json",
    data_files="hf://datasets/miracl/miracl-corpus/"
               "miracl-corpus-v1.0-ru/docs-*.jsonl.gz",
    split="train", streaming=True)
texts: list[str] = []
for row in stream:
    t = row["text"].strip()
    if 100 <= len(t) <= 1000:          # Отсекаем слишком короткие и длинные
        texts.append(t)
    if len(texts) >= 10_050:
        break

CORPUS = texts[:10_000]                # 10 000 пассажей — корпус
QUERIES = texts[10_000:10_050]         # 50 пассажей — запросы
assert len(CORPUS) == 10_000 and len(QUERIES) == 50


def embed_e5_docs(model, texts):
    """Эмбеддинги документов через multilingual-e5: к каждому
    документу спереди добавляется префикс 'passage: '."""
    inputs = ["passage: " + t for t in texts]
    return model.encode(inputs, normalize_embeddings=True,
                        batch_size=64, show_progress_bar=True)


def embed_e5_queries(model, texts):
    """Эмбеддинги запросов через multilingual-e5: к каждому запросу
    спереди добавляется префикс 'query: '."""
    inputs = ["query: " + t for t in texts]
    return model.encode(inputs, normalize_embeddings=True,
                        batch_size=64, show_progress_bar=True)


# --- Считаем эмбеддинги; FAISS требует непрерывный массив float32 ---
e5 = SentenceTransformer("intfloat/multilingual-e5-base", device="cuda")
corpus_vecs = np.ascontiguousarray(embed_e5_docs(e5, CORPUS), dtype="float32")
query_vecs = np.ascontiguousarray(embed_e5_queries(e5, QUERIES), dtype="float32")
d = corpus_vecs.shape[1]               # Размерность вектора e5-base: 768
print(f"Корпус: {corpus_vecs.shape}, запросы: {query_vecs.shape}")


# --- Строим три индекса ---

# 1. Точный индекс: хранит float32 как есть, обучение не нужно
flat = faiss.IndexFlatIP(d)
flat.add(corpus_vecs)

# 2. Scalar quantization: int8, требует обучения для калибровки диапазонов
# Пропуск 1: создайте IndexScalarQuantizer (QT_8bit, METRIC_INNER_PRODUCT),
# обучите его на corpus_vecs и добавьте векторы
sq = faiss.IndexScalarQuantizer(
    d,
    faiss.ScalarQuantizer.QT_8bit,
    faiss.METRIC_INNER_PRODUCT
)
sq.train(corpus_vecs)
sq.add(corpus_vecs)

# 3. Product Quantization: M=96 подвекторов, требует обучения кодовых книг
# Пропуск 2: создайте IndexPQ(d, M, 8, METRIC_INNER_PRODUCT),
# обучите его на corpus_vecs и добавьте векторы
M = 96                                 # 768 / 96 = 8 координат в подвекторе
pq = faiss.IndexPQ(
    d,
    M,
    8,
    faiss.METRIC_INNER_PRODUCT
)
pq.train(corpus_vecs)
pq.add(corpus_vecs)


def index_size_mb(index, path):
    """Пишет индекс на диск и возвращает размер файла в мегабайтах."""
    # Пропуск 3: сохраните индекс через faiss.write_index(index, path)
    # и верните размер файла в МБ через os.path.getsize(path)
    faiss.write_index(index, path)
    return os.path.getsize(path) / 1e6


def search_time(index, queries, k=5):
    """Время поиска top-k для всех запросов, в секундах."""
    t0 = time.perf_counter()
    index.search(queries, k)
    return time.perf_counter() - t0


print(f"\n{'Индекс':<24}{'Размер, МБ':>12}{'Поиск 50 зпр., с':>20}")
for name, index in [("IndexFlatIP", flat),
                    ("IndexScalarQuantizer", sq),
                    (f"IndexPQ (M={M})", pq)]:
    size = index_size_mb(index, f"{name}.faiss")
    secs = search_time(index, query_vecs, k=5)
    print(f"{name:<24}{size:>12.2f}{secs:>20.4f}")

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.64it/s]


Корпус: (10000, 768), запросы: (50, 768)

Индекс                    Размер, МБ    Поиск 50 зпр., с
IndexFlatIP                    30.72              0.0229
IndexScalarQuantizer            7.69              0.0255
IndexPQ (M=96)                  1.75              0.0129


In [4]:
import os
import faiss
import numpy as np

# corpus_vecs, query_vecs, d, flat, sq — из Задания 4;
#  index_size_mb() — оттуда же


def recall_at_k(approx_index, exact_index, queries, k=5):
    """Доля top-k точного поиска (эталон IndexFlat), которую
    воспроизводит сжатый индекс. Усреднение по всем запросам.
    approx_index — сжатый индекс, exact_index — точный (ground truth)."""
    _, exact = exact_index.search(queries, k)    # Эталонные top-k, форма (n, k)
    _, approx = approx_index.search(queries, k)  # Top-k сжатого индекса
    hits = 0
    for exact_row, approx_row in zip(exact, approx):
        # Пропуск 1: пересечение эталонной и приближённой пятёрок
        common = set(exact_row) & set(approx_row)
        hits += len(common)
    # Пропуск 2: recall = все попадания / (число запросов * k)
    return hits / (len(queries) * k)


# int8 уже построен в Задании 4 как sq; считаем его размер и recall
print(f"{'Метод':<26}{'Сжатие':>8}{'Размер, МБ':>12}{'recall@5':>12}")
sq_size = index_size_mb(sq, "sq.faiss")
print(f"{'IndexScalarQuantizer':<26}{'4x':>8}"
      f"{sq_size:>12.2f}{recall_at_k(sq, flat, query_vecs):>12.3f}")

# PQ с разными M: проходим шкалу сжатия от мягкого к жёсткому
for M in [384, 192, 96, 48]:
    # Пропуск 3: создайте IndexPQ(d, M, 8, METRIC_INNER_PRODUCT),
    # обучите на corpus_vecs и добавьте векторы
    pq = faiss.IndexPQ(
        d,
        M,
        8,
        faiss.METRIC_INNER_PRODUCT
    )
    pq.train(corpus_vecs)
    pq.add(corpus_vecs)

    ratio = (768 * 4) // M                        # Сжатие по кодам: байт было/стало
    size = index_size_mb(pq, f"pq_{M}.faiss")
    r = recall_at_k(pq, flat, query_vecs)
    print(f"{'IndexPQ (M=' + str(M) + ')':<26}"
          f"{str(ratio) + 'x':>8}{size:>12.2f}{r:>12.3f}")

Метод                       Сжатие  Размер, МБ    recall@5
IndexScalarQuantizer            4x        7.69       0.976
IndexPQ (M=384)                 8x        4.63       0.928
IndexPQ (M=192)                16x        2.71       0.700
IndexPQ (M=96)                 32x        1.75       0.524
IndexPQ (M=48)                 64x        1.27       0.272
